# 🥇 Gold Layer — Discount Analysis
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/silver/` and `medallion/gold/`, enriches order-level discount data with classification flags and customer LTV metrics, and writes to `medallion/gold/`.

| Source Tables | Gold Table | Key Transformations |
|---|---|---|
| `gold_customer_orders`, `gold_customer_profiles`, `silver_order_discounts_lookup` | `gold_discount_analysis` | Classify discount type, flag high-magnitude codes, flag B2B/affiliate orders, flag stacked discounts, merge customer LTV & retention, select final columns |

## 0. Setup & Paths

In [1]:
import pandas as pd
import numpy as np
import os
import re
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = Path.cwd()
SILVER_DIR = BASE / 'medallion' / 'silver'
GOLD_DIR   = BASE / 'medallion' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base   : {BASE}")
print(f"Silver : {SILVER_DIR}")
print(f"Gold   : {GOLD_DIR}")

Base   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505
Silver : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/silver
Gold   : /Users/bennyteo/Desktop/Benny/MITB/Term 3/ISSS603 Applied Data Science for Customer Insights/Project/LushProtein_Project_Data_20260505/medallion/gold


---
## 1. Discount Analysis (`gold_customer_orders` → `gold_discount_analysis`)

### Step 1 — Load data & start from `gold_customer_orders`

In [2]:
# Load data
co = pd.read_parquet(GOLD_DIR / 'gold_customer_orders.parquet')
cp = pd.read_parquet(GOLD_DIR / 'gold_customer_profiles.parquet')
discount_lookup = pd.read_parquet(SILVER_DIR / 'silver_order_discounts_lookup.parquet')

# Start from gold_customer_orders (one row per order)
df = co.copy()

### Step 2 — Classify discount type

In [3]:
# Classify discount type
def classify_discount(name):
    if pd.isna(name): return None
    name_l = name.lower()
    if any(x in name_l for x in ['shopee', 'lazada', 'seller voucher',
                                   'transaction fee', 'commission']):
        return 'marketplace_fee'
    if any(x in name_l for x in ['free gift', 'freegift']):
        return 'free_gift'
    if any(x in name_l for x in ['buy 2', 'buy 4', '2 packs',
                                   'single', 'line item', '4 packs']):
        return 'bundle_discount'
    if 'custom discount' in name_l:
        return 'custom'
    return 'promo_code'

df['discount_type'] = df['discount_code'].apply(classify_discount)

### Step 3 — Flag high-magnitude discount codes (≥30% off)

In [4]:
# Flag high-magnitude discount codes (≥30% off)
# Based on known codes — can be updated once business confirms thresholds
high_magnitude_codes = [
    'BDAY40', 'BF40', 'SG40', 'MAY30', 'FWE30', 'FWESG2025', 'SGFWE2025',
    'BLACKFRIDAY', 'september', '3030', 'SALE30', 'JUNE20', '20OFF', 'SALE20'
]
df['is_high_magnitude'] = (
    df['discount_code'].str.upper().isin([c.upper() for c in high_magnitude_codes])
)

### Step 4 — Flag B2B and affiliate orders

In [5]:
# Flag B2B and affiliate orders (exclude from retail analysis)
b2b_affiliate_codes = ['B2B100', 'SGAFF100', 'affiliate', 'JESCHIN']
df['is_b2b_or_affiliate'] = (
    df['discount_code'].str.upper().isin([c.upper() for c in b2b_affiliate_codes]) |
    df['discount_code'].str.lower().str.contains('aff', na=False)
)

### Step 5 — Flag discount stacking (multiple codes on one order)

In [6]:
# Flag discount stacking (multiple codes on one order) 
df['is_stacked_discount'] = df['discount_code'].str.contains(',', na=False)

### Step 6 — Merge customer LTV and retention from `gold_customer_profiles`

In [7]:
# Merge customer LTV and retention from gold_customer_profiles 
df = df.merge(
    cp[['customer_id', 'total_orders', 'total_revenue', 'avg_order_value',
        'repeat_purchase_90d', 'rfm_group', 'is_repeat_customer']],
    on='customer_id', how='left'
)

### Step 7 — Select final columns & save

In [8]:
# Select final columns
gold_discount_analysis = df[[
    'customer_id', 'order_id', 'order_name', 'processed_at',
    'channel', 'country_code',
    'price_total', 'price_total_discount',
    'discount_code', 'discount_amount', 'discount_type',
    'is_high_magnitude', 'is_b2b_or_affiliate', 'is_stacked_discount',
    'order_sequence', 'is_first_order',
    'total_orders', 'total_revenue', 'avg_order_value',
    'repeat_purchase_90d', 'rfm_group', 'is_repeat_customer'
]].copy()

print(f"gold_discount_analysis: {gold_discount_analysis.shape[0]:,} rows × {gold_discount_analysis.shape[1]} cols")
print(f"\nDiscount type breakdown:")
print(gold_discount_analysis['discount_type'].value_counts(dropna=False).to_string())
print(f"\nHigh magnitude orders: {gold_discount_analysis['is_high_magnitude'].sum():,}")
print(f"B2B/affiliate orders: {gold_discount_analysis['is_b2b_or_affiliate'].sum():,}")
print(f"Stacked discount orders: {gold_discount_analysis['is_stacked_discount'].sum():,}")
print(f"\nFirst order discount rate: {gold_discount_analysis[gold_discount_analysis['is_first_order']]['discount_code'].notna().mean():.1%}")

gold_discount_analysis.to_parquet(GOLD_DIR / 'gold_discount_analysis.parquet', index=False)
print("\n✅ Saved: gold_discount_analysis.parquet")

gold_discount_analysis: 28,054 rows × 22 cols

Discount type breakdown:
discount_type
None               19557
promo_code          3942
marketplace_fee     2230
bundle_discount     1949
free_gift            268
custom               108

High magnitude orders: 1,603
B2B/affiliate orders: 232
Stacked discount orders: 2,112

First order discount rate: 34.1%

✅ Saved: gold_discount_analysis.parquet


#### Discount type breakdown

None (19,557 / 69.7%) — orders with no discount applied

promo_code (3,942 / 14.1%) — actual promotional codes like BDAY40, BF40, MAY30 — the core group for Q2 and Q7 analysis

marketplace_fee (2,230 / 7.9%) — Shopee/Lazada platform fees recorded as discounts, not real customer discounts — exclude from Q2/Q7

bundle_discount (1,949 / 6.9%) — automatic buy 2/4 pack deals

free_gift (268 / 1.0%) — free product additions

custom (108 / 0.4%) — manually applied discounts